In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState


class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command


@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
    )

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='7fa21835-8ff9-4141-8040-016169ab7013'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 283, 'prompt_tokens': 141, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EACo7YOPYECvWOF5pJNP438obBkAL', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdbed-bb3e-7802-9359-c644143f2cb4-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_hWt5lvrRRqXH4XgaeSP5bhq7', 'type': 'tool_call'}], invalid_

In [7]:
pprint(response["messages"][-1].content)

('Nice. I’ve noted that green is your favourite colour. Want me to tailor '
 'suggestions or themes around green, or is there something else you’d like to '
 'update?')


In [ ]:
message = HumanMessage(content="Hello, how are you?")

response = agent.invoke(
    {
        "messages": [message],
        "favourite_colour": "green"
    }, # type: ignore
    {"configurable": {"thread_id": "10"}}
    )

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='d7b6f7be-b432-4b35-928d-d2c39a8b3422'),
              AIMessage(content='Hi there! I’m an AI, so I don’t have feelings, but I’m here and ready to help with whatever you need. How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 364, 'prompt_tokens': 142, 'total_tokens': 506, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAEFKpnae591kdAzG582LwM2w4Ubz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fdc42-1bab-7ee1-8a95-1b00116eb383-0', tool_calls=[], invalid_tool_calls=[], usage

In [14]:
pprint(response["messages"][-1].content)

('I’m here and ready to help. I don’t have feelings, but I’m functioning well. '
 'What would you like to do or chat about today?')


## Read state

In [15]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

In [16]:
agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
    )

In [17]:
message = HumanMessage(content="My favourite colour is green")

response = agent.invoke(
    { "messages": [message]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='f03d3113-e6f3-48d0-ac73-c314876c1ab7'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 411, 'prompt_tokens': 162, 'total_tokens': 573, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EAEVfuVa5X0bfiRoEmpwkyWozJde4', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdc51-91df-7663-ac9b-b0b43ec4aff6-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_02mQ3Nw2qjmevC8FsXLh98QY', 'type': 'tool_call'}], invalid_

In [18]:
pprint(response["messages"][-1].content)

('Nice! I’ve saved your favourite colour as green. Would you like me to tailor '
 'things around green (e.g., color palettes, themes, or prompts)?')


In [19]:
message = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    { "messages": [message]},
    {"configurable": {"thread_id": "1"}}
)

In [20]:
pprint(response["messages"][-1].content)

('Your favourite colour is green.\n'
 '\n'
 'Would you like me to tailor things around green, such as color palettes, '
 'themes, or prompts? I can generate green-inspired palettes or ideas if you’d '
 'like.')
